# TF-IDF

En este cuadernillo se construye un sistema de recomendación basado en contenido, usando TF-IDF para vectorizar el texto de cada película y calculando después la similitud del coseno entre esos vectores.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Importación de los conjuntos de datos
Se cargan los conjuntos de ratings (train, val y test) y el dataset de PLN (NLP.parquet) con la información de cada película. Train y val se unen en un único conjunto de entrenamiento.

In [ ]:
t =  pd.read_parquet("../../data/03_model_ready/ratings_train.parquet")
test =  pd.read_parquet("../../data/03_model_ready/ratings_test.parquet")
val =  pd.read_parquet("../../data/03_model_ready/ratings_val.parquet")
train = pd.concat([t, val], ignore_index=True)
df = pd.read_parquet("../../data/03_model_ready/NLP.parquet")

## Preprocesamiento de datos
Se limpia la columna overview, eliminando los caracteres que no sean letras y pasando el texto a minúsculas.

In [3]:
df['overview'] = df['overview'].str.replace(r'[^a-zA-Z\s]', '', regex=True).str.lower() 

Se utiliza spaCy para el preprocesamiento del texto: se carga el modelo en_core_web_sm, entrenado para inglés.

In [4]:
# hay que añadir uv add click
# es necesario descargar el modelo: uv run python -m spacy download en_core_web_sm
# carga modelo preentrenado.
import spacy 
nlp = spacy.load("en_core_web_sm") 

La función preprocesamiento pasa el texto a minúsculas, elimina las stopwords (palabras muy frecuentes y poco informativas, como artículos o preposiciones) y sustituye cada palabra por su lema.

In [5]:
def preprocesamiento(texto):
    doc = nlp(texto.lower()) #minuscula y objeto doc
    tokens = [
        token.lemma_
        for token in doc
            if not token.is_stop and token.lemma_.strip() != ""]
    return " ".join(tokens)

In [6]:
import spacy 
nlp = spacy.load("en_core_web_sm") 

Se recarga df desde NLP.parquet, ordenado por movieId, y se aplica la función preprocesamiento a la columna join (que ya contiene genres, tag y overview unidos) para obtener el texto que se usará en el TF-IDF.

In [7]:
df = pd.read_parquet("../../data/03_model_ready/NLP.parquet").sort_values('movieId')
df.head()

,movieId,title,genres,tag,overview,join
0,1,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ...",adventure animation children comedy fantasy pi...
1,2,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...,adventure children fantasy fantasy magic board...
2,3,Grumpier Old Men,comedy romance,moldy old,A family wedding reignites the ancient feud be...,comedy romance moldy old A family wedding reig...
3,4,Waiting to Exhale,comedy drama romance,,"Cheated on, mistreated and stepped on, the wom...","comedy drama romance Cheated on, mistreated a..."
4,5,Father of the Bride Part II,comedy,pregnancy remake,Just when George Banks has recovered from his ...,comedy pregnancy remake Just when George Banks...


In [8]:
df["texto_TFIDF"] = df["join"].apply(preprocesamiento)

In [9]:
df.head()

,movieId,title,genres,tag,overview,join,texto_TFIDF
0,1,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ...",adventure animation children comedy fantasy pi...,adventure animation child comedy fantasy pixar...
1,2,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...,adventure children fantasy fantasy magic board...,adventure child fantasy fantasy magic board ga...
2,3,Grumpier Old Men,comedy romance,moldy old,A family wedding reignites the ancient feud be...,comedy romance moldy old A family wedding reig...,comedy romance moldy old family wedding reigni...
3,4,Waiting to Exhale,comedy drama romance,,"Cheated on, mistreated and stepped on, the wom...","comedy drama romance Cheated on, mistreated a...","comedy drama romance cheat , mistreat step , w..."
4,5,Father of the Bride Part II,comedy,pregnancy remake,Just when George Banks has recovered from his ...,comedy pregnancy remake Just when George Banks...,comedy pregnancy remake george bank recover da...


## Vectorización TF-IDF

Se utilizan los conjuntos train y test. El de validación ya está incluido en train y no hace falta usarlo aparte.

Se calcula el TF-IDF del texto preprocesado (texto_TFIDF): cada película queda representada como un vector, con una coordenada por cada palabra del vocabulario.

In [10]:
#corpus de documentos 
corpus = df['texto_TFIDF'].tolist()

# objeto 
vectorizer = TfidfVectorizer( ) # max_features=5000,min_df=5,max_df=0.8

# fit aprende el vocabulario completo y transform vectoriza cada documento
# se tiene una coordenada por cada palabra del vocabulario
X = vectorizer.fit_transform(corpus)

# muestra las palabras que forman el vocabulario
vectorizer.get_feature_names_out()

array(['00', '000', '007', ..., 'žižek', 'ʻohana', 'сhatterer'],
      shape=(23288,), dtype=object)

## Perfil de usuario
Primer intento de construir el perfil de usuario como media ponderada por rating de los vectores TF-IDF de las películas vistas. Este código queda comentado porque, al convertir la matriz TF-IDF a densa con toarray(), no cabe en memoria.

In [11]:
# df_tfidf_content = pd.DataFrame(
#     X.toarray(),
#     columns=vectorizer.get_feature_names_out(),
#     index=df['movieId']
# )
# df_tfidf_content
# datos = pd.merge(df_tfidf_content, train, on = 'movieId', how = 'right')

# generos = vectorizer.get_feature_names_out().tolist()

# # hay peliculas que no tienen genero asociado
# datos[generos] = datos[generos].fillna(0)

# #usando la media ponderada por pesos(ratings). usuario es DF
# def agregacion(usuario):
#     vectores= usuario[generos].values
#     pesos    = usuario['rating'].values
#     return np.average(vectores, axis = 0, weights=pesos) # 0 es para que sea por columna

# df_tfidf_BasedProfile = datos.groupby('userId').apply(agregacion).reset_index() 

# df_tfidf_BasedProfile.columns=['userId', 'BasedProfile']
# df_tfidf_BasedProfile

# no entra en memoria

Se repite la misma idea, pero manteniendo la matriz X en formato sparse (sin convertirla a densa), calculando la media ponderada por rating directamente sobre los vectores dispersos. Así el perfil de cada usuario se calcula sin agotar la memoria.

In [12]:
# from scipy import sparse
import numpy as np

corpus = vectorizer.get_feature_names_out()

# X ya es sparse (la salida de TfidfVectorizer)
# Crear un mapping de movieId a índice en X
movie_to_idx = {mid: i for i, mid in enumerate(df['movieId'])}

perfiles = []
for user_id, group in train.groupby('userId'):
    indices = [movie_to_idx[m] for m in group['movieId'] if m in movie_to_idx]
    if indices:
        vectores = X[indices]  # sigue siendo sparse, no consume memoria
        pesos = group.loc[group['movieId'].isin(movie_to_idx), 'rating'].values
        # Media ponderada: sum(vector_i * peso_i) / sum(pesos)
        perfil = np.array((vectores.T @ pesos) / pesos.sum()).flatten()
    else:
        perfil = np.zeros(len(corpus))
    perfiles.append({'userId': user_id, 'BasedProfile': perfil})

df_tfidf_BasedProfile = pd.DataFrame(perfiles)

In [13]:
len(corpus)

23288

## Recomendación
Para recomendar a un usuario, se calcula la similitud del coseno entre su perfil y los vectores TF-IDF de todas las películas. Se excluyen las películas ya vistas y se devuelven las k con mayor similitud. La función imprime además las películas que el usuario ya ha visto, para poder comparar.

In [14]:
def recomendacionUser(userId, k=10):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    vistas = df[df['movieId'].isin(pelisVistas)][['movieId', 'title','genres']]
    print(vistas)
    return pelisRec

Ejemplo de recomendación para el usuario 2: primero se listan las películas que ha visto, y después las 10 recomendadas.

In [15]:
# recomendacion para un usuario
recomendacionUser(2)


      movieId                                         title  \
277       318                    Shawshank Redemption, The    
291       333                                    Tommy Boy    
2669     3578                                    Gladiator    
4605     6874                            Kill Bill: Vol. 1    
5288     8798                                   Collateral    
6218    46970  Talladega Nights: The Ballad of Ricky Bobby    
6280    48516                                Departed, The    
6669    58559                             Dark Knight, The    
6760    60756                                Step Brothers    
6960    68157                         Inglourious Basterds    
7100    71535                                   Zombieland    
7203    74458                               Shutter Island    
7267    77455                   Exit Through the Gift Shop    
7316    79132                                    Inception    
7379    80906                                   Inside 

,movieId,title,similitud
257,296,Pulp Fiction,0.164756
2221,2959,Fight Club,0.164087
254,293,Léon: The Professional (a.k.a. The Professiona...,0.157849
5884,33794,Batman Begins,0.155852
8479,116419,Drive Hard,0.143660
3554,4878,Donnie Darko,0.142328
509,592,Batman,0.139690
8895,139385,The Revenant,0.137247
4596,6860,Mobsters,0.135262
9060,148626,"Big Short, The",0.134988


## Evaluación de las recomendaciones
Para cada usuario se generan las k=10 recomendaciones y se comparan con las películas de test que ha valorado. Se considera relevante una película de test si su rating es mayor o igual que el umbral t (por defecto 3.5). Con esto se calculan tres métricas por usuario:
- precisionK: proporción de las k recomendaciones que resultan relevantes.
- recalK: proporción de las películas relevantes de test que aparecen entre las recomendaciones.
- F1K: media armónica de precisionK y recalK.

resultadosUmbral calcula estas métricas para todos los usuarios y devuelve su media (multiplicada por p, para expresar el resultado en porcentaje).

In [16]:
def evaluacion(userId, k=10, t=3.5):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    datos = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'right')
    interseccion = datos[~datos['rating'].isna()] #peliculas vistas y que han sido recomendadas

    nRelevantes = (interseccion['rating'] >= t).sum()
    nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    precisionk = nRelevantes/k
    if nTest !=0:
        recalk = nRelevantes/nTest
    else:
        recalk = 0
    if recalk + precisionk != 0:
        F1k = 2*recalk*precisionk/(recalk + precisionk)
    else:
        F1k = 0
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'precisionK': [precisionk],
        'recalK':[recalk],
        'F1K':[F1k]
        })
    return resultados
    
def resultadosUmbral(t, p=100): # resultados en %
    listaUs = train['userId'].unique()
    evaluacionTFIDF = pd.DataFrame()
    for i in listaUs:
        evaluacionTFIDF = pd.concat([evaluacionTFIDF,evaluacion(i,t=t)], ignore_index=True)

    solucion = evaluacionTFIDF[['precisionK', 'recalK','F1K']].mean()*p 
    return solucion


Resultado con t=3.5 y p=100 (valor por defecto), es decir, las métricas expresadas como porcentaje.

In [17]:
resultadosUmbral(3.5) 

precisionK    0.901639
recalK        1.086769
F1K           0.788104
dtype: float64

## Evaluación alternativa: solo usuarios con intersección no vacía
En la evaluación anterior, muchos usuarios no tienen ninguna coincidencia entre sus recomendaciones y sus películas de test, por lo que su métrica se queda en 0 y penaliza la media global. Aquí se prueba una variante que solo tiene en cuenta a los usuarios cuya intersección entre recomendaciones y test no está vacía, para medir el rendimiento del modelo únicamente donde sí ha encontrado coincidencias.

In [18]:
def evaluacionMOD(userId, k=10, t=3.5):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    interseccion = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'inner')
    nRelevantes = (interseccion['rating'] >= t).sum()

    # nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    if len(interseccion)==0:
        metrica = 0
        inter = 0
    else:
        metrica = nRelevantes/len(interseccion)
        inter = 1

  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'interseccion':[inter],
        'metrica':[metrica]
        })
    return resultados
    
def resultadosUmbralMOD(t, p=100):
    listaUs = train['userId'].unique()
    evaluacionTFIDF = pd.DataFrame()
    for i in listaUs:
        evaluacionTFIDF = pd.concat([evaluacionTFIDF,evaluacionMOD(i,t=t)], ignore_index=True)

    solucion = evaluacionTFIDF[evaluacionTFIDF['metrica'] != 0.0][[ 'metrica']]
    print(f"Número de usuarios son intersección no vacía: {len(solucion)}")
    return solucion.mean()*p

Resultado con t=3.5: se muestra cuántos usuarios tienen intersección no vacía y la métrica media (en porcentaje) entre esos usuarios.

In [19]:
resultadosUmbralMOD(3.5)

Número de usuarios son intersección no vacía: 51


metrica    98.039216
dtype: float64